## LIBRARIES

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## WIDGETS

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:

dbutils.widgets.text("storageName", "saccexplorer")
dbutils.widgets.text("containerName", "bronze")
dbutils.widgets.text("catalogName", "unit_catalog_explorer")
dbutils.widgets.text("schemaName", "uc_bronze")


## CONSTANTS

In [0]:
storage = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("containerName")
catalog =  dbutils.widgets.get("catalogName")
schema =  dbutils.widgets.get("schemaName")

## PATHS

In [0]:
path_base_bronze = f"abfss://{container}@{storage}.dfs.core.windows.net/{schema}"

path_sector   = f"{path_base_bronze}/supercias_sector"

## SOURCES

In [0]:
url_sector = "https://appscvsmovil.supercias.gob.ec/ranking/recursos/bi_ciiu.csv"

## STRUCTURES

In [0]:
# import pandas as pd
# df_sector = pd.read_csv(url_sector)
# print(df_sector.dtypes)

sector_schema = StructType([
    StructField("ciiu", StringType(), True),
    StructField("descripcion", StringType(), True)
])

## READ SOURCE

In [0]:
# --- Descarga del archivo (Lectura directa desde el driver)
import requests

temp_local_path = "/tmp/sector_temp.csv"
response = requests.get(url_sector, verify=False) 

if response.status_code == 200:
    with open(temp_local_path, "wb") as f:
        f.write(response.content)
    print("Archivo descargado localmente en el Driver.")
else:
    raise Exception(f"Error al descargar: {response.status_code}")


# %sh ls -lh /tmp/


## SAVE SOURCE

In [0]:
# --- Lectura con Spark
df_sector = (spark.read
              .option("header", "True")
              .option("delimiter", ",") 
              .schema(sector_schema)
              .csv(f"file:{temp_local_path}")) 

# --- Escritura a Delta (Unity Catalog)
df_sector.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_sector) \
    .saveAsTable(f"{catalog}.{schema}.supercias_sector")

print(f"¡Éxito! Tabla creada en {catalog}.{schema}.supercias_sector")


# --- BLOQUE DE LIMPIEZA ---
import os
if os.path.exists(temp_local_path):
    os.remove(temp_local_path)
    print(f"Archivo temporal {temp_local_path} eliminado del Driver.")


# %sh ls -lh /tmp/